In [ ]:
# Importación de Librerías
import pandas as pd 

from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score
from lightgbm import LGBMClassifier, early_stopping

def metric(y_true, y_pred):
    res = cohen_kappa_score(y_true, y_pred.reshape((y_true.shape[0], 5), order="F").argmax(axis=1), weights= 'quadratic')
    return "kappa", res, True

In [ ]:
train = pd.read_csv('../input/petfinder-adoption-prediction/train/train.csv').set_index("PetID")
test = pd.read_csv('../input/petfinder-adoption-prediction/test/test.csv').set_index("PetID")
folds = [train.index[test_idx] for train_idx, test_idx in KFold(5).split(train)]

In [ ]:
# EXP: guardar Name antes de que el baseline lo descarte con select_dtypes
nombres_train = train["Name"].copy()
nombres_test = test["Name"].copy()

In [ ]:
for c in ['Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2',
       'Color3', 'FurLength', 'MaturitySize', 'Vaccinated', 'Dewormed',
       'Sterilized', 'Health', 'State', 'RescuerID']:
    train[c] = train[c].astype("category")    
    test[c] = test[c].astype("category")

In [ ]:
train = train\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/train.parquet"))\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-img/train.parquet"))\
    .select_dtypes(exclude="O")
train

In [ ]:
test = test\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/test.parquet"))\
    .join(pd.read_parquet("../input/examen-lab-ii-preprocesado-img/test.parquet"))\
    .select_dtypes(exclude="O")
test

In [ ]:
fold_preds = []
test_preds = []
for i, fold in enumerate(folds):
    X_train = train.drop(fold).drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_train = train.drop(fold)["AdoptionSpeed"]

    X_valid = train.loc[fold].drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_valid = train.loc[fold, "AdoptionSpeed"]

    learner = LGBMClassifier()
    learner.fit(X_train, y_train, )
    
    learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                eval_metric= metric, callbacks=[early_stopping(5)])
    fold_preds.append(pd.Series(learner.predict(X_valid), index=fold, name="AdoptionSpeed"))
    test_preds.append(pd.DataFrame(learner.predict_proba(test[X_train.columns]), index=test.index))

fold_preds = pd.concat(fold_preds)
test_preds = sum(test_preds).idxmax(axis=1).rename("AdoptionSpeed")

In [ ]:
# EXP-003: baseline ya corrido arriba; ahora la variante con sin_nombre
pred_base = fold_preds

for df, nombres in ((train, nombres_train), (test, nombres_test)):
    nombre = nombres.fillna("").str.lower().str.replace(" ", "", regex=False)
    df["sin_nombre"] = ((nombre == "") | nombre.str.startswith("noname")).astype(int)

In [ ]:
def correr(train, test):
    fold_preds = []
    test_preds = []
    importancias = []
    for i, fold in enumerate(folds):
        X_train = train.drop(fold).drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
        y_train = train.drop(fold)["AdoptionSpeed"]

        X_valid = train.loc[fold].drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
        y_valid = train.loc[fold, "AdoptionSpeed"]

        learner = LGBMClassifier()
        learner.fit(X_train, y_train, )
        
        learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                    eval_metric= metric, callbacks=[early_stopping(5)])
        fold_preds.append(pd.Series(learner.predict(X_valid), index=fold, name="AdoptionSpeed"))
        test_preds.append(pd.DataFrame(learner.predict_proba(test[X_train.columns]), index=test.index))
        importancias.append(pd.Series(learner.feature_importances_, index=X_train.columns))

    fold_preds = pd.concat(fold_preds)
    test_preds = sum(test_preds).idxmax(axis=1).rename("AdoptionSpeed")
    return fold_preds, test_preds, importancias

In [ ]:
pred_exp, test_preds, imp_exp = correr(train, test)

In [ ]:
def qwk_por_fold(fold_preds):
    return [cohen_kappa_score(train.loc[f, "AdoptionSpeed"], fold_preds.loc[f], weights="quadratic") for f in folds]

res = pd.DataFrame({"baseline": qwk_por_fold(pred_base), "exp_003": qwk_por_fold(pred_exp)},
                   index=[f"fold_{i}" for i in range(len(folds))])
res["delta"] = res["exp_003"] - res["baseline"]
print(res.round(4).to_string())
print("media baseline", round(res["baseline"].mean(), 4), "desvio", round(res["baseline"].std(), 4))
print("media exp_003 ", round(res["exp_003"].mean(), 4), "desvio", round(res["exp_003"].std(), 4))
print("delta media   ", round(res["delta"].mean(), 4))
print("folds que mejoran:", int((res["delta"] > 0).sum()), "de", len(folds))
print("importancia (split) de sin_nombre por fold:", [int(s["sin_nombre"]) for s in imp_exp])

In [ ]:
test_preds.to_csv("submission.csv")
!head submission.csv